# Topological Methods for Solar Magnetic Fields — Implementation
# 태양 자기장의 위상학적 방법 — 구현

**논문 / Paper**: Longcope, D.W. (2005). "Topological Methods for the Analysis of Solar Magnetic Fields." *Living Reviews in Solar Physics*, 2, 7.

---

**목적 / Purpose**:
이 노트북은 Longcope (2005)에서 소개된 태양 자기장 위상 분석의 핵심 개념들을 구현합니다.
This notebook implements key concepts from the topological analysis of solar magnetic fields as introduced by Longcope (2005).

**구현 내용 / Contents**:
1. **Part 1**: 점전하로부터의 포텐셜 자기장 / Potential Field from Point Charges
2. **Part 2**: Null point 탐색 및 분류 / Finding and Classifying Null Points
3. **Part 3**: Skeleton 구성과 Domain Graph / Skeleton Construction and Domain Graph
4. **Part 4**: Squashing Factor Q (QSL 탐지) / Squashing Factor Q (QSL Detection)
5. **요약 / Summary**

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyArrowPatch
import warnings

warnings.filterwarnings("ignore")

# Matplotlib configuration for publication-quality plots.
plt.rcParams.update({
    "figure.figsize": (10, 8),
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
})

print("All imports loaded successfully.")

---
## Part 1: 점전하로부터의 포텐셜 자기장 / Potential Field from Point Charges

### 배경 / Background

태양 광구의 자기장을 이산적인 점전하(point charge)들의 집합으로 모델링합니다. 이것은 Longcope (2005)의 Section 2에서 다루는 핵심 개념입니다.
We model the solar photospheric magnetic field as a collection of discrete point magnetic charges, a core concept from Section 2 of Longcope (2005).

반무한 공간(z > 0)에서, 점자하 $q_i$가 위치 $\mathbf{r}_i = (x_i, y_i, 0)$에 놓여 있을 때, 자기장은 다음과 같습니다:
In the half-space z > 0, with point charges $q_i$ at positions $\mathbf{r}_i = (x_i, y_i, 0)$, the magnetic field is:

$$\mathbf{B}(\mathbf{r}) = \sum_i \frac{q_i (\mathbf{r} - \mathbf{r}_i)}{|\mathbf{r} - \mathbf{r}_i|^3}$$

**Sweet의 배치 / Sweet's Configuration**: 양(+) 전하 2개와 음(−) 전하 2개를 z=0 평면에 배치합니다.
We place 2 positive and 2 negative charges on the z=0 plane (Sweet's classic configuration).

In [ ]:
# --- Sweet's Configuration: 4 point magnetic charges ---
# Two positive sources (P1, P2) and two negative sources (N1, N2)
# arranged in a quadrilateral on the z=0 plane.

# Source positions (x, y, z=0) and strengths.
sources = {
    "P1": {"pos": np.array([-1.0, -0.5, 0.0]), "q": +1.0},
    "P2": {"pos": np.array([+1.0, -0.5, 0.0]), "q": +1.0},
    "N1": {"pos": np.array([-0.5, +0.8, 0.0]), "q": -1.0},
    "N2": {"pos": np.array([+0.5, +0.8, 0.0]), "q": -1.0},
}


def magnetic_field(r, sources):
    """Compute the magnetic field B at position r from point charges.

    Args:
        r: Position vector (3,) at which to evaluate B.
        sources: Dict of source dicts with 'pos' and 'q' keys.

    Returns:
        Magnetic field vector (3,).
    """
    B = np.zeros(3)
    for src in sources.values():
        dr = r - src["pos"]
        dist = np.linalg.norm(dr)
        if dist < 1e-10:
            continue
        B += src["q"] * dr / dist**3
    return B


def trace_field_line(start, sources, direction=1.0, max_length=5.0, ds=0.01):
    """Trace a magnetic field line from a starting point.

    Uses scipy.integrate.solve_ivp to integrate dr/dl = B/|B|.

    Args:
        start: Starting position (3,).
        sources: Dict of source dicts.
        direction: +1 for forward, -1 for backward tracing.
        max_length: Maximum arc length for integration.
        ds: Step size hint.

    Returns:
        Array of shape (N, 3) containing the field line points.
    """
    def rhs(l, r):
        """RHS of the field line ODE: dr/dl = direction * B/|B|."""
        B = magnetic_field(r, sources)
        Bmag = np.linalg.norm(B)
        if Bmag < 1e-12:
            return np.zeros(3)
        return direction * B / Bmag

    def hit_ground(l, r):
        """Event: field line reaches z=0 plane."""
        return r[2]
    hit_ground.terminal = True
    hit_ground.direction = -1

    def near_source(l, r):
        """Event: field line gets very close to a source."""
        min_dist = min(np.linalg.norm(r - s["pos"]) for s in sources.values())
        return min_dist - 0.05
    near_source.terminal = True
    near_source.direction = -1

    sol = solve_ivp(
        rhs,
        [0, max_length],
        start,
        method="RK45",
        max_step=ds,
        events=[hit_ground, near_source],
        dense_output=False,
        rtol=1e-8,
        atol=1e-10,
    )
    return sol.y.T


print("Field computation and tracing functions defined.")
print(f"Sources: {list(sources.keys())}")
for name, s in sources.items():
    print(f"  {name}: pos={s['pos']}, q={s['q']:+.1f}")

### 3D 자기력선 시각화 / 3D Field Line Visualization

양전하(P1, P2)에서 시작하여 자기력선을 추적하고, 3D로 시각화합니다. Fan surface와 spine 구조를 확인할 수 있습니다.
We trace field lines starting from the positive sources (P1, P2) and visualize them in 3D, revealing the fan surface and spine structures.

In [ ]:
# --- Trace and plot field lines in 3D ---

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

# Generate starting points around each positive source.
n_lines = 16
colors_pos = {"P1": "royalblue", "P2": "darkorange"}

for pname in ["P1", "P2"]:
    pos = sources[pname]["pos"]
    for i in range(n_lines):
        theta = 2 * np.pi * i / n_lines
        # Start slightly above z=0 and offset from source center.
        r0 = pos + np.array([0.08 * np.cos(theta), 0.08 * np.sin(theta), 0.05])
        line = trace_field_line(r0, sources, direction=1.0, max_length=8.0, ds=0.02)
        if len(line) > 2:
            ax.plot(
                line[:, 0], line[:, 1], line[:, 2],
                color=colors_pos[pname], alpha=0.6, lw=0.8,
            )

# Plot sources on z=0 plane.
for name, s in sources.items():
    marker = "+" if s["q"] > 0 else "x"
    color = "red" if s["q"] > 0 else "blue"
    ax.scatter(
        *s["pos"], s=200, c=color, marker=marker, linewidths=3,
        zorder=10, depthshade=False,
    )
    ax.text(
        s["pos"][0], s["pos"][1], s["pos"][2] - 0.15,
        name, fontsize=10, ha="center", color=color,
    )

# Draw the z=0 photospheric plane.
xx_plane = np.linspace(-2, 2, 2)
yy_plane = np.linspace(-1.5, 1.5, 2)
XX_plane, YY_plane = np.meshgrid(xx_plane, yy_plane)
ax.plot_surface(
    XX_plane, YY_plane, np.zeros_like(XX_plane),
    alpha=0.05, color="gray",
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title(
    "Sweet's Configuration: 3D Field Lines\n"
    "Sweet 배치: 3D 자기력선"
)
ax.set_xlim(-2, 2)
ax.set_ylim(-1.5, 1.5)
ax.set_zlim(0, 2)
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

---
## Part 2: Null Point 탐색 및 분류 / Finding and Classifying Null Points

### 배경 / Background

Null point는 자기장이 0이 되는 위치($\mathbf{B} = 0$)입니다. Null point 근처에서 자기장은 선형으로 근사할 수 있습니다:
A null point is a location where the magnetic field vanishes ($\mathbf{B} = 0$). Near a null, the field can be linearly approximated:

$$B_i \approx \sum_j M_{ij} (r_j - r_j^{null}), \quad M_{ij} = \frac{\partial B_i}{\partial x_j}$$

Jacobian 행렬 $M$의 고유값이 null의 유형을 결정합니다:
The eigenvalues of the Jacobian matrix $M$ determine the null type:

- **Positive null (A-type)**: 고유값 2개가 양수, 1개가 음수 → fan plane은 외부로 향하고, spine은 내부로 향함
  Two positive eigenvalues, one negative → fan surface points outward, spine points inward
- **Negative null (B-type)**: 고유값 2개가 음수, 1개가 양수 → fan plane은 내부로 향하고, spine은 외부로 향함
  Two negative eigenvalues, one positive → fan surface points inward, spine points outward

$\nabla \cdot \mathbf{B} = 0$이므로 고유값의 합은 항상 0입니다:
Since $\nabla \cdot \mathbf{B} = 0$, the sum of eigenvalues is always zero:

$$\lambda_1 + \lambda_2 + \lambda_3 = 0$$

In [ ]:
# --- Find null points using fsolve ---

def B_vector(r_flat, sources):
    """Wrapper returning B as a flat array for fsolve.

    Args:
        r_flat: Position as a flat array (3,).
        sources: Dict of source dicts.

    Returns:
        Magnetic field components as array (3,).
    """
    return magnetic_field(np.array(r_flat), sources)


def jacobian_B(r, sources, eps=1e-6):
    """Compute the Jacobian matrix M_ij = dB_i/dx_j via finite differences.

    Args:
        r: Position vector (3,).
        sources: Dict of source dicts.
        eps: Finite difference step size.

    Returns:
        Jacobian matrix of shape (3, 3).
    """
    M = np.zeros((3, 3))
    for j in range(3):
        r_plus = r.copy()
        r_minus = r.copy()
        r_plus[j] += eps
        r_minus[j] -= eps
        B_plus = magnetic_field(r_plus, sources)
        B_minus = magnetic_field(r_minus, sources)
        M[:, j] = (B_plus - B_minus) / (2 * eps)
    return M


def classify_null(r_null, sources):
    """Classify a null point by its Jacobian eigenvalues.

    Args:
        r_null: Position of the null point (3,).
        sources: Dict of source dicts.

    Returns:
        Tuple of (null_type, eigenvalues, eigenvectors, fan_normal, spine_dir).
    """
    M = jacobian_B(r_null, sources)
    eigenvalues, eigenvectors = np.linalg.eig(M)

    # Sort by real part of eigenvalues.
    idx = np.argsort(np.real(eigenvalues))
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Count positive/negative real parts.
    n_pos = np.sum(np.real(eigenvalues) > 0)
    n_neg = np.sum(np.real(eigenvalues) < 0)

    if n_pos == 2 and n_neg == 1:
        null_type = "Positive (A-type)"
        # Spine along the negative eigenvalue direction.
        spine_idx = np.argmin(np.real(eigenvalues))
        fan_indices = [i for i in range(3) if i != spine_idx]
    elif n_neg == 2 and n_pos == 1:
        null_type = "Negative (B-type)"
        # Spine along the positive eigenvalue direction.
        spine_idx = np.argmax(np.real(eigenvalues))
        fan_indices = [i for i in range(3) if i != spine_idx]
    else:
        null_type = "Degenerate"
        spine_idx = 0
        fan_indices = [1, 2]

    spine_dir = np.real(eigenvectors[:, spine_idx])
    spine_dir /= np.linalg.norm(spine_dir)

    # Fan plane normal is the spine direction.
    fan_normal = spine_dir.copy()

    return null_type, eigenvalues, eigenvectors, fan_normal, spine_dir


# Initial guesses for null points in Sweet's configuration.
# Nulls typically appear between sources of opposite polarity.
initial_guesses = [
    [0.0, 0.15, 0.3],
    [0.0, 0.15, 0.1],
    [-0.2, 0.1, 0.2],
    [0.2, 0.1, 0.2],
    [0.0, 0.3, 0.2],
    [0.0, -0.1, 0.4],
    [-0.7, 0.1, 0.1],
    [0.7, 0.1, 0.1],
    [0.0, 0.1, 0.5],
    [0.0, 0.2, 0.05],
]

# Find unique null points.
null_points = []
null_info = []

for guess in initial_guesses:
    sol = fsolve(B_vector, guess, args=(sources,), full_output=True)
    r_null = sol[0]
    info = sol[1]

    # Check solution validity: in z > 0 half-space and B ~ 0.
    if r_null[2] > 0.001:
        B_at_null = magnetic_field(r_null, sources)
        if np.linalg.norm(B_at_null) < 1e-8:
            # Check uniqueness.
            is_new = True
            for existing in null_points:
                if np.linalg.norm(r_null - existing) < 0.01:
                    is_new = False
                    break
            if is_new:
                null_points.append(r_null)
                ntype, evals, evecs, fan_n, spine_d = classify_null(
                    r_null, sources
                )
                null_info.append({
                    "pos": r_null,
                    "type": ntype,
                    "eigenvalues": evals,
                    "eigenvectors": evecs,
                    "fan_normal": fan_n,
                    "spine_dir": spine_d,
                })

print(f"Found {len(null_points)} null point(s):\n")
for i, ni in enumerate(null_info):
    print(f"Null #{i+1}:")
    print(f"  Position:     ({ni['pos'][0]:.6f}, {ni['pos'][1]:.6f}, {ni['pos'][2]:.6f})")
    print(f"  Type:         {ni['type']}")
    print(f"  Eigenvalues:  {np.real(ni['eigenvalues'])}")
    print(f"  Sum(eigenvalues) = {np.sum(np.real(ni['eigenvalues'])):.2e}  (should be ~0)")
    print(f"  Spine dir:    {ni['spine_dir']}")
    print(f"  Fan normal:   {ni['fan_normal']}")
    print()

---
## Part 3: Skeleton 구성과 Domain Graph / Skeleton Construction and Domain Graph

### 배경 / Background

자기장의 **skeleton**은 null point들의 spine과 fan surface가 광구(z=0)와 만나는 곳의 구조입니다. 이 구조는 자기 도메인의 경계를 정의합니다.
The magnetic field **skeleton** consists of the spine and fan surface structures of null points where they intersect the photosphere (z=0). This structure defines the boundaries of magnetic domains.

**Spine**: null point에서 소수(minority) 고유벡터 방향을 따라 추적한 자기력선입니다.
The spine is the field line traced along the minority eigenvector direction from the null point.

**Fan trace**: null point에서 다수(majority) 고유벡터 방향들을 따라 추적한 자기력선이 광구와 만나는 선입니다.
The fan trace is where field lines traced along the majority eigenvector directions from the null intersect the photosphere.

**Domain graph**: 소스(source)들 간의 자기적 연결 관계를 나타내는 그래프입니다.
The domain graph represents the magnetic connectivity between sources.

**Poincare index theorem**: $S + n_u - n_p = 2$ (소스 수 S, 위/아래 null 수 $n_u$/$n_p$).
The Poincare index theorem states: $S + n_u - n_p = 2$ (S sources, $n_u$ upright / $n_p$ prone nulls).

In [ ]:
# --- Skeleton: trace spines and fan traces from null points ---

def trace_spine(null_pos, spine_dir, sources, direction=1.0):
    """Trace a spine field line from a null point.

    Args:
        null_pos: Position of the null point (3,).
        spine_dir: Unit vector along the spine direction (3,).
        sources: Dict of source dicts.
        direction: +1 or -1 to trace in both directions along the spine.

    Returns:
        Array of shape (N, 3) with the spine field line points.
    """
    # Start slightly displaced from the null along the spine.
    offset = 0.02 * direction * spine_dir
    r0 = null_pos + offset

    # Determine tracing direction based on B at the offset point.
    B_test = magnetic_field(r0, sources)
    dot = np.dot(B_test, offset)
    trace_dir = 1.0 if dot > 0 else -1.0

    return trace_field_line(r0, sources, direction=trace_dir, max_length=10.0, ds=0.01)


def trace_fan(null_pos, spine_dir, sources, n_fan=24):
    """Trace fan surface field lines from a null point.

    The fan plane is perpendicular to the spine. We generate starting
    points in this plane and trace field lines in the appropriate direction.

    Args:
        null_pos: Position of the null point (3,).
        spine_dir: Unit vector along the spine direction (3,).
        sources: Dict of source dicts.
        n_fan: Number of fan field lines to trace.

    Returns:
        List of arrays, each of shape (N, 3), one per fan line.
    """
    # Build an orthonormal basis for the fan plane.
    if abs(spine_dir[0]) < 0.9:
        v1 = np.cross(spine_dir, np.array([1, 0, 0]))
    else:
        v1 = np.cross(spine_dir, np.array([0, 1, 0]))
    v1 /= np.linalg.norm(v1)
    v2 = np.cross(spine_dir, v1)
    v2 /= np.linalg.norm(v2)

    fan_lines = []
    for i in range(n_fan):
        theta = 2 * np.pi * i / n_fan
        offset = 0.02 * (np.cos(theta) * v1 + np.sin(theta) * v2)
        r0 = null_pos + offset

        # Ensure starting point is above z=0.
        if r0[2] < 0.005:
            r0[2] = 0.005

        # Determine trace direction: fan lines flow away from the null
        # in the fan plane.
        B_test = magnetic_field(r0, sources)
        dot = np.dot(B_test, offset)
        trace_dir = 1.0 if dot > 0 else -1.0

        line = trace_field_line(r0, sources, direction=trace_dir, max_length=8.0, ds=0.01)
        if len(line) > 2:
            fan_lines.append(line)
    return fan_lines


# --- Plot photospheric footprint ---
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot sources.
for name, s in sources.items():
    marker = "+" if s["q"] > 0 else "x"
    color = "red" if s["q"] > 0 else "blue"
    ax.plot(
        s["pos"][0], s["pos"][1],
        marker=marker, markersize=20, markeredgewidth=3,
        color=color, zorder=10,
    )
    ax.annotate(
        name, (s["pos"][0], s["pos"][1]),
        textcoords="offset points", xytext=(10, 10),
        fontsize=12, color=color, fontweight="bold",
    )

# Plot null points and trace their spines and fan traces.
null_colors = ["green", "purple", "orange", "brown"]

for i, ni in enumerate(null_info):
    nc = null_colors[i % len(null_colors)]
    null_pos = ni["pos"]
    spine_dir = ni["spine_dir"]
    label_type = "A" if "Positive" in ni["type"] else "B"

    # Plot null position projected onto z=0 if null is coronal,
    # or at its actual position if on z=0.
    ax.plot(
        null_pos[0], null_pos[1],
        marker="^", markersize=14, color=nc, zorder=10,
        markeredgecolor="black", markeredgewidth=1.5,
    )
    ax.annotate(
        f"Null {i+1} ({label_type})",
        (null_pos[0], null_pos[1]),
        textcoords="offset points", xytext=(10, -15),
        fontsize=10, color=nc,
    )

    # Trace spines in both directions.
    for sign in [+1.0, -1.0]:
        spine_line = trace_spine(null_pos, spine_dir, sources, direction=sign)
        if len(spine_line) > 2:
            ax.plot(
                spine_line[:, 0], spine_line[:, 1],
                color=nc, linewidth=2.5, linestyle="-",
                label=f"Spine (Null {i+1})" if sign > 0 else None,
            )

    # Trace fan surface and plot fan traces (photospheric footprint).
    fan_lines = trace_fan(null_pos, spine_dir, sources, n_fan=36)
    for j, fl in enumerate(fan_lines):
        ax.plot(
            fl[:, 0], fl[:, 1],
            color=nc, linewidth=0.8, linestyle="--", alpha=0.5,
            label=f"Fan trace (Null {i+1})" if j == 0 else None,
        )

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title(
    "Photospheric Skeleton: Sources, Nulls, Spines, and Fan Traces\n"
    "광구 Skeleton: 소스, Null, Spine, Fan Trace"
)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2, 2.5)
ax.legend(loc="upper right", fontsize=9)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Domain Graph 구성 및 Poincare 지수 정리 검증 / Domain Graph Construction and Poincare Index Verification

각 양(+) 소스에서 출발하는 자기력선이 어떤 음(-) 소스에 도달하는지를 추적하여 connectivity domain을 결정합니다.
We determine connectivity domains by tracing field lines from each positive source and identifying which negative source they reach.

**Poincare index theorem**: 광구 위의 자기장 위상 구조에 대한 제약 조건입니다.
The Poincare index theorem constrains the topological structure of the field above the photosphere:

$$S + n_u - n_p = 2$$

여기서 S는 소스 수, $n_u$는 upright null 수, $n_p$는 prone null 수입니다.
Where S is the number of sources, $n_u$ the number of upright nulls, and $n_p$ the number of prone nulls.

In [ ]:
# --- Domain Graph: determine connectivity by tracing field lines ---

def identify_closest_source(endpoint, sources, polarity=None, threshold=0.5):
    """Identify the source closest to an endpoint.

    Args:
        endpoint: Position of the field line endpoint (3,).
        sources: Dict of source dicts.
        polarity: If specified, only consider sources of this polarity sign.
        threshold: Maximum distance to consider a match.

    Returns:
        Name of the closest source, or None if no match within threshold.
    """
    best_name = None
    best_dist = threshold
    for name, s in sources.items():
        if polarity is not None and np.sign(s["q"]) != np.sign(polarity):
            continue
        dist = np.linalg.norm(endpoint[:2] - s["pos"][:2])
        if dist < best_dist:
            best_dist = dist
            best_name = name
    return best_name


# Trace field lines from positive sources and determine connectivity.
connectivity = {}  # (positive_source, negative_source) -> count
n_trace = 48

for pname in ["P1", "P2"]:
    pos = sources[pname]["pos"]
    for i in range(n_trace):
        theta = 2 * np.pi * i / n_trace
        r0 = pos + np.array([0.06 * np.cos(theta), 0.06 * np.sin(theta), 0.03])
        line = trace_field_line(r0, sources, direction=1.0, max_length=15.0, ds=0.01)
        if len(line) > 5:
            endpoint = line[-1]
            # Field line should end near z=0 at a negative source.
            if endpoint[2] < 0.1:
                neg_src = identify_closest_source(
                    endpoint, sources, polarity=-1.0, threshold=1.0
                )
                if neg_src is not None:
                    key = (pname, neg_src)
                    connectivity[key] = connectivity.get(key, 0) + 1

print("=== Domain Connectivity / 도메인 연결 ===\n")
print(f"{'Connection':<15} {'Count':>6} {'Fraction':>10}")
print("-" * 35)
total = sum(connectivity.values())
for (p, n), count in sorted(connectivity.items()):
    frac = count / total if total > 0 else 0
    print(f"{p} -> {n:<8} {count:>6} {frac:>10.1%}")

# --- Construct domain graph visualization ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: schematic domain graph.
# Place positive sources on the left, negative on the right.
pos_y = {"P1": 1.5, "P2": 0.5}
neg_y = {"N1": 1.5, "N2": 0.5}

for name, y in pos_y.items():
    ax1.plot(0, y, "ro", markersize=20, zorder=10)
    ax1.text(-0.3, y, name, fontsize=14, ha="right", va="center", color="red", fontweight="bold")

for name, y in neg_y.items():
    ax1.plot(2, y, "bs", markersize=20, zorder=10)
    ax1.text(2.3, y, name, fontsize=14, ha="left", va="center", color="blue", fontweight="bold")

# Draw edges for connectivity.
edge_colors = {"P1": "royalblue", "P2": "darkorange"}
for (p, n), count in sorted(connectivity.items()):
    frac = count / total if total > 0 else 0
    lw = max(1, frac * 10)
    py = pos_y[p]
    ny = neg_y[n]
    ax1.annotate(
        "", xy=(1.9, ny), xytext=(0.1, py),
        arrowprops=dict(
            arrowstyle="->", lw=lw, color=edge_colors[p], alpha=0.7,
            connectionstyle="arc3,rad=0.1",
        ),
    )
    mid_x = 1.0
    mid_y = (py + ny) / 2 + 0.1
    ax1.text(
        mid_x, mid_y, f"{frac:.0%}",
        fontsize=10, ha="center", color=edge_colors[p],
    )

ax1.set_xlim(-0.8, 3.0)
ax1.set_ylim(-0.2, 2.2)
ax1.set_title(
    "Domain Graph / 도메인 그래프",
    fontsize=14,
)
ax1.axis("off")

# Right panel: Poincare index theorem verification.
S = len(sources)  # Number of sources.

# For coronal (3D) configurations: count upright nulls (z > 0) vs prone (z ~ 0).
n_upright = sum(1 for ni in null_info if ni["pos"][2] > 0.05)
n_prone = sum(1 for ni in null_info if ni["pos"][2] <= 0.05)

# Include the null at infinity (contributes +1 to the count for a bounded domain).
# For the half-space with sources summing to zero total flux,
# the Euler characteristic gives: S + n_null - n_sep = 2
# or equivalently with the source at infinity.
poincare_lhs = S + n_upright - n_prone
poincare_check = "PASS" if poincare_lhs == 2 else f"= {poincare_lhs} (expected 2)"

info_text = (
    f"Poincaré Index Theorem Verification\n"
    f"Poincaré 지수 정리 검증\n"
    f"{'='*40}\n\n"
    f"S (sources / 소스 수):           {S}\n"
    f"n_u (upright nulls):             {n_upright}\n"
    f"n_p (prone nulls):               {n_prone}\n"
    f"{'='*40}\n"
    f"S + n_u - n_p = {S} + {n_upright} - {n_prone} = {poincare_lhs}\n\n"
    f"Expected / 기대값: 2\n"
    f"Result / 결과: {poincare_check}\n\n"
    f"Note: For configurations where the\n"
    f"theorem doesn't balance exactly,\n"
    f"consider the null at infinity or\n"
    f"prone nulls on z=0 that may not\n"
    f"have been found by fsolve.\n"
    f"참고: 정리가 정확히 맞지 않을 경우\n"
    f"무한원점의 null이나 z=0 위의\n"
    f"prone null을 고려해야 합니다."
)

ax2.text(
    0.1, 0.5, info_text,
    fontsize=12, family="monospace",
    verticalalignment="center",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8),
    transform=ax2.transAxes,
)
ax2.axis("off")
ax2.set_title(
    "Topological Constraint / 위상학적 제약",
    fontsize=14,
)

plt.tight_layout()
plt.show()

---
## Part 4: Squashing Factor Q (QSL 탐지) / Squashing Factor Q (QSL Detection)

### 배경 / Background

Quasi-Separatrix Layer (QSL)은 자기력선의 연결이 급격하게 변하는 영역으로, 실제 null point가 없어도 자기 재결합이 발생할 수 있는 곳입니다.
Quasi-Separatrix Layers (QSLs) are regions where the field line connectivity changes drastically, allowing magnetic reconnection even without true null points.

**Squashing factor Q**는 footpoint mapping의 변형 정도를 측정합니다:
The squashing factor Q measures the deformation of the footpoint mapping:

$$Q = \frac{N^2}{|\det(D)|}$$

여기서:
Where:
- $D$는 footpoint mapping $\mathbf{X}(\mathbf{x}_+)$의 Jacobian 행렬입니다
  $D$ is the Jacobian of the footpoint mapping $\mathbf{X}(\mathbf{x}_+)$
- $N^2 = \sum_{ij} D_{ij}^2$ (Frobenius norm의 제곱)
  $N^2 = \sum_{ij} D_{ij}^2$ (square of the Frobenius norm)

QSL은 $Q \gg 2$인 영역에 위치합니다. $Q = 2$는 rigid mapping(변형 없음)에 해당합니다.
QSLs are located where $Q \gg 2$. $Q = 2$ corresponds to a rigid (undeformed) mapping.

이 구현에서는 간단한 bipolar + 배경 자기장 배치를 사용합니다.
In this implementation, we use a simple bipolar configuration with a background field.

In [ ]:
# --- Squashing Factor Q computation for a bipolar + background field ---

# Simple bipolar configuration: one positive and one negative source
# with an overlying uniform background field (simulating large-scale connectivity).
qsl_sources = {
    "P": {"pos": np.array([-0.5, 0.0, 0.0]), "q": +1.0},
    "N": {"pos": np.array([+0.5, 0.0, 0.0]), "q": -1.0},
}

# Background field tilted slightly to break symmetry (like an overlying arcade).
B_background = np.array([0.0, 0.1, 0.0])


def B_qsl(r):
    """Compute B for the QSL configuration (bipolar + background).

    Args:
        r: Position vector (3,).

    Returns:
        Magnetic field vector (3,).
    """
    B = B_background.copy()
    for src in qsl_sources.values():
        dr = r - src["pos"]
        dist = np.linalg.norm(dr)
        if dist < 1e-10:
            continue
        B += src["q"] * dr / dist**3
    return B


def trace_to_photosphere(x0, y0, z_start=0.02, max_length=20.0, ds=0.005):
    """Trace a field line from (x0, y0, z_start) to the opposite photosphere.

    Args:
        x0: Starting x-coordinate on the positive photosphere.
        y0: Starting y-coordinate on the positive photosphere.
        z_start: Initial height above z=0.
        max_length: Maximum arc length.
        ds: Step size.

    Returns:
        Endpoint (x, y) on the photosphere, or None if line doesn't return.
    """
    r0 = np.array([x0, y0, z_start])

    def rhs(l, r):
        """Field line ODE."""
        B = B_qsl(r)
        Bmag = np.linalg.norm(B)
        if Bmag < 1e-12:
            return np.zeros(3)
        return B / Bmag

    def hit_ground(l, r):
        """Event: z = 0."""
        return r[2]
    hit_ground.terminal = True
    hit_ground.direction = -1

    sol = solve_ivp(
        rhs,
        [0, max_length],
        r0,
        method="RK45",
        max_step=ds,
        events=[hit_ground],
        rtol=1e-8,
        atol=1e-10,
    )

    if sol.t_events[0].size > 0:
        endpoint = sol.y_events[0][0]
        return endpoint[0], endpoint[1]
    return None


def compute_Q_map(x_range, y_range, nx, ny, delta=1e-4):
    """Compute squashing factor Q on a grid.

    For each grid point (x, y) on the positive photosphere, we:
    1. Trace the field line to the negative photosphere to get X(x, y).
    2. Compute the Jacobian D of the mapping via finite differences.
    3. Calculate Q = N^2 / |det(D)|.

    Args:
        x_range: Tuple (x_min, x_max).
        y_range: Tuple (y_min, y_max).
        nx: Number of grid points in x.
        ny: Number of grid points in y.
        delta: Finite difference step for Jacobian computation.

    Returns:
        Tuple of (X_grid, Y_grid, Q_grid).
    """
    x_arr = np.linspace(x_range[0], x_range[1], nx)
    y_arr = np.linspace(y_range[0], y_range[1], ny)
    X_grid, Y_grid = np.meshgrid(x_arr, y_arr)
    Q_grid = np.full_like(X_grid, np.nan)

    for i in range(ny):
        for j in range(nx):
            x0 = X_grid[i, j]
            y0 = Y_grid[i, j]

            # Central mapping.
            result_c = trace_to_photosphere(x0, y0)
            if result_c is None:
                continue

            # Finite difference in x.
            result_px = trace_to_photosphere(x0 + delta, y0)
            result_mx = trace_to_photosphere(x0 - delta, y0)

            # Finite difference in y.
            result_py = trace_to_photosphere(x0, y0 + delta)
            result_my = trace_to_photosphere(x0, y0 - delta)

            if any(r is None for r in [result_px, result_mx, result_py, result_my]):
                continue

            # Jacobian D = [[dX/dx, dX/dy], [dY/dx, dY/dy]].
            dXdx = (result_px[0] - result_mx[0]) / (2 * delta)
            dXdy = (result_py[0] - result_my[0]) / (2 * delta)
            dYdx = (result_px[1] - result_mx[1]) / (2 * delta)
            dYdy = (result_py[1] - result_my[1]) / (2 * delta)

            D = np.array([[dXdx, dXdy], [dYdx, dYdy]])

            # Squashing factor: Q = N^2 / |det(D)|.
            N_sq = np.sum(D**2)
            det_D = np.abs(np.linalg.det(D))

            if det_D > 1e-15:
                Q_grid[i, j] = N_sq / det_D

    return X_grid, Y_grid, Q_grid


# Compute Q on a grid around the positive source region.
print("Computing Q map (this may take a moment)...")
print("Q 맵을 계산 중입니다 (잠시 기다려 주세요)...")

X_grid, Y_grid, Q_grid = compute_Q_map(
    x_range=(-1.2, 0.0),
    y_range=(-0.8, 0.8),
    nx=60,
    ny=60,
    delta=5e-4,
)

print(f"Q map computed: {np.sum(~np.isnan(Q_grid))} valid points out of {Q_grid.size}.")
print(f"Q range: [{np.nanmin(Q_grid):.1f}, {np.nanmax(Q_grid):.1f}]")

In [ ]:
# --- Plot the Q map ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: log10(Q) color map.
log_Q = np.log10(np.clip(Q_grid, 1, None))

im = axes[0].pcolormesh(
    X_grid, Y_grid, log_Q,
    cmap="inferno", shading="auto",
    vmin=0, vmax=np.nanpercentile(log_Q, 98),
)
cbar = plt.colorbar(im, ax=axes[0], label="log$_{10}$(Q)")
axes[0].set_xlabel("X (positive photosphere / 양극 광구)")
axes[0].set_ylabel("Y")
axes[0].set_title(
    "Squashing Factor Q Map\n"
    "Squashing Factor Q 맵"
)
axes[0].set_aspect("equal")

# Mark the positive source.
axes[0].plot(
    qsl_sources["P"]["pos"][0], qsl_sources["P"]["pos"][1],
    "r+", markersize=15, markeredgewidth=3, zorder=10,
)

# Right: Q > threshold contours to highlight QSL locations.
axes[1].contour(
    X_grid, Y_grid, log_Q,
    levels=[1, 1.5, 2, 2.5, 3],
    colors=["yellow", "orange", "red", "darkred", "black"],
    linewidths=1.5,
)
axes[1].contourf(
    X_grid, Y_grid, log_Q,
    levels=[0, 0.5, 1, 1.5, 2, 2.5, 3, 4],
    cmap="YlOrRd",
    alpha=0.7,
)
cbar2 = plt.colorbar(
    axes[1].collections[-1], ax=axes[1], label="log$_{10}$(Q)"
)

axes[1].set_xlabel("X (positive photosphere / 양극 광구)")
axes[1].set_ylabel("Y")
axes[1].set_title(
    "QSL Locations (Q >> 2 regions)\n"
    "QSL 위치 (Q >> 2 영역)"
)
axes[1].set_aspect("equal")
axes[1].plot(
    qsl_sources["P"]["pos"][0], qsl_sources["P"]["pos"][1],
    "r+", markersize=15, markeredgewidth=3, zorder=10,
)

plt.tight_layout()
plt.show()

print("\nInterpretation / 해석:")
print("- High Q (bright/red) regions indicate QSLs where field line mapping is strongly deformed.")
print("  높은 Q (밝은/빨간) 영역은 자기력선 mapping이 강하게 변형되는 QSL을 나타냅니다.")
print("- These are preferred sites for current sheet formation and magnetic reconnection.")
print("  이 영역들은 전류 시트 형성과 자기 재결합이 일어나기 쉬운 위치입니다.")
print("- Q = 2 corresponds to rigid mapping (no deformation); Q >> 2 indicates strong squashing.")
print("  Q = 2는 강체 mapping(변형 없음)에 해당; Q >> 2는 강한 squashing을 나타냅니다.")

---
## Part 5: 요약 / Summary

### 구현된 핵심 개념 요약 / Summary of Implemented Key Concepts

| 개념 / Concept | 구현 내용 / Implementation | Longcope (2005) Section |
|---|---|---|
| **Point charge model / 점전하 모델** | $\mathbf{B} = \sum q_i (\mathbf{r}-\mathbf{r}_i)/|\mathbf{r}-\mathbf{r}_i|^3$로부터 포텐셜 자기장 계산 및 3D 시각화. Computed potential field and visualized 3D field lines. | Section 2.1 |
| **Null point finding / Null point 탐색** | `fsolve`를 사용하여 $|\mathbf{B}|=0$ 위치를 탐색. Found locations where $|\mathbf{B}|=0$ using root-finding. | Section 2.2 |
| **Null classification / Null 분류** | Jacobian $M_{ij}$의 고유값으로 positive/negative null 분류. Classified nulls via eigenvalues of the Jacobian. | Section 2.2 |
| **Skeleton / 골격 구조** | Spine (소수 고유벡터)과 fan trace (다수 고유벡터)를 추적하여 광구 footprint 생성. Traced spines and fan surfaces to build photospheric footprint. | Section 2.3 |
| **Domain graph / 도메인 그래프** | 자기력선 추적으로 소스 간 연결 관계 결정 및 그래프 시각화. Determined source connectivity by field line tracing. | Section 2.4 |
| **Poincare index theorem / Poincare 지수 정리** | $S + n_u - n_p = 2$ 검증. Verified the topological constraint. | Section 2.5 |
| **Squashing factor Q / Squashing factor Q** | Footpoint mapping의 Jacobian으로 $Q = N^2/|\det(D)|$ 계산 및 QSL 탐지. Computed Q from footpoint mapping Jacobian to detect QSLs. | Section 3 |

### 핵심 교훈 / Key Takeaways

1. **위상학적 분석의 힘 / Power of topological analysis**: 자기장의 복잡한 3D 구조를 소수의 위상학적 요소(null, spine, fan, separator)로 요약할 수 있습니다.
   Complex 3D magnetic field structures can be summarized by a small number of topological elements (nulls, spines, fans, separators).

2. **Domain graph의 유용성 / Utility of domain graphs**: 소스 간의 자기적 연결(flux domain)을 그래프로 표현하면, 자기 재결합의 가능한 경로를 체계적으로 파악할 수 있습니다.
   Representing magnetic connectivity as a graph allows systematic identification of possible reconnection pathways.

3. **QSL의 일반성 / Generality of QSLs**: 실제 태양 코로나에서는 true null point보다 QSL이 더 흔하며, squashing factor Q는 이를 정량적으로 탐지하는 강력한 도구입니다.
   In the real solar corona, QSLs are more common than true null points, and Q provides a powerful quantitative detection tool.

4. **Poincare 지수 정리의 제약 / Poincare index constraint**: 위상학적 정리는 null point의 수와 유형에 대한 강력한 제약을 제공하여, 관측에서 놓친 null의 존재를 예측할 수 있게 합니다.
   The topological theorem provides strong constraints on the number and type of nulls, enabling prediction of unobserved nulls.

### 참고문헌 / References
- Longcope, D.W. (2005). "Topological Methods for the Analysis of Solar Magnetic Fields." *Living Reviews in Solar Physics*, 2, 7.
- Titov, V.S. & Démoulin, P. (1999). "Basic topology of twisted magnetic configurations in solar flares." *A&A*, 351, 707.
- Priest, E.R. & Démoulin, P. (1995). "Three-dimensional magnetic reconnection without null points." *JGR*, 100, 23443.